In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [26]:
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [27]:
#preprocess data
#remove irrelevant data
data=data.drop(['RowNumber','CustomerId','Surname'], axis=1)

In [28]:
#encode categorical variable
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [29]:
#one hot encode geography
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder()
geo_encoded=ohe.fit_transform(data[['Geography']])
geo_encoded=geo_encoded.toarray()

In [30]:
geo_df= pd.DataFrame(
  geo_encoded,
  columns=ohe.get_feature_names_out(['Geography'])

)
data=pd.concat([data, geo_df],axis=1)



In [31]:
data=data.drop(['Geography'], axis=1)

In [32]:
#divide data into independent and dependent data

X=data.drop('EstimatedSalary',axis=1)
y=data['EstimatedSalary']

In [33]:
#split train test
X_train,X_test,y_train, y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [34]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [35]:
#save encoders and scalers
with open('label_encoder_gender.pkl','wb') as file:
  pickle.dump(label_encoder_gender, file)

with open('ohe.pkl', 'wb') as file:
  pickle.dump(ohe, file)

with open('scaler.pkl', 'wb') as file:
  pickle.dump(scaler, file)


## ANN Regression


In [36]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime


In [37]:
#build ANN model
model=Sequential([
  Dense(64, activation='relu', input_shape=(X_train.shape[1],)), #first hidden layer connected with input layer
  Dense(32, activation='relu'),#hidden layer 2
  Dense(1)#output layer
]

)

In [38]:
model.compile(optimizer='adam',loss='mean_absolute_error', metrics=['mae'])

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                832       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [39]:
#set up tensor board
import datetime
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir="regressionlogs/fit"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir, histogram_freq=1)



In [40]:
## set up early stopping where loss not decreasing anymore
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)
 

In [41]:
#training the model
history=model.fit(
  X_train, y_train, validation_data=(X_test, y_test), epochs=100,
  callbacks=[tensorflow_callback, early_stopping_callback]
)

Epoch 1/100
250/250 [==============================] - 1s 2ms/step - loss: 100371.4219 - mae: 100371.4219 - val_loss: 98493.7031 - val_mae: 98493.7031
Epoch 2/100
250/250 [==============================] - 0s 2ms/step - loss: 99546.7344 - mae: 99546.7344 - val_loss: 96829.3438 - val_mae: 96829.3438
Epoch 3/100
250/250 [==============================] - 0s 2ms/step - loss: 96627.9141 - mae: 96627.9141 - val_loss: 92492.6719 - val_mae: 92492.6719
Epoch 4/100
250/250 [==============================] - 0s 2ms/step - loss: 90834.9844 - mae: 90834.9844 - val_loss: 85335.6328 - val_mae: 85335.6328
Epoch 5/100
250/250 [==============================] - 0s 2ms/step - loss: 82587.8281 - mae: 82587.8281 - val_loss: 76287.8047 - val_mae: 76287.8047
Epoch 6/100
250/250 [==============================] - 0s 2ms/step - loss: 73124.0703 - mae: 73124.0703 - val_loss: 67264.4844 - val_mae: 67264.4844
Epoch 7/100
250/250 [==============================] - 0s 2ms/step - loss: 64359.7578 - mae: 64359.7578 

In [42]:
# load tensorboard extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [43]:
%tensorboard --logdir regressionlogs/fit --port 6008





Reusing TensorBoard on port 6008 (pid 24964), started 2:08:57 ago. (Use '!kill 24964' to kill it.)

In [44]:
#evaluate model on the test data
test_loss, test_mae=model.evaluate(X_test, y_test)
print(f'Test MAE: {test_mae}')

63/63 [==============================] - 0s 1ms/step - loss: 50300.7656 - mae: 50300.7656
Test MAE: 50300.765625


In [45]:
train_loss, train_mae = model.evaluate(X_train, y_train)
print("Train MAE:", train_mae)

250/250 [==============================] - 0s 1ms/step - loss: 49515.8359 - mae: 49515.8359
Train MAE: 49515.8359375


In [46]:
model.save('regression_model.h5')

d:\conda_envs\DL1\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
